# Single-particle validation: Euclidean vs polar coordinates

This notebook compares the same physical problem integrated in two coordinate systems:

- `euclidean_solver_v1.py`: physical Cartesian coordinates `(x, y)`.
- `polar_solver_v2.py`: polar coordinates `(r, theta)`, with physical contacts and a Cartesian cap near `r = 0`.

The physical test is one particle moving under gravity inside a circular soft wall. The polar trajectory is mapped back to Cartesian space before comparison.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from euclidean_solver_v1 import simulate_euclidean_particles
from polar_solver_v2 import (
    simulate_polar_particles,
    euclidean_to_polar_particles,
)

plt.rcParams.update({
    "figure.figsize": (7, 5),
    "axes.grid": True,
})

## Problem definition

The wall is circular and fixed here (`xA = 0`). This keeps the validation focused on the coordinate transformation and geometric acceleration terms.

In [ ]:
radius = 20.0
particle_radius = 0.20
particle_mass = 1.0

x0 = np.array([[0.0, 1.0]], dtype=np.float32)
v0 = np.array([[10.0, 20.0]], dtype=np.float32)

m = np.array([particle_mass], dtype=np.float32)
rad = np.array([particle_radius], dtype=np.float32)
group_mobile = np.array([0, 1], dtype=np.int64)

# This is a float32 DEM-style solver. Making dt extremely small, for example
# 1e-7 over many seconds, is not a route to higher accuracy: roundoff from
# tens of millions of updates eventually dominates the truncation error.
dt = 1.0e-4
T_max = 10.0
save_every = 20

g = np.array([0.0, -9.81], dtype=np.float32)
xA = np.array([0.0, 0.0], dtype=np.float32)
omega = 0.0

# High stiffness makes the soft wall close to reflective while keeping the
# same force law in both solvers.
k_w = 2.0e5
gamma_w = 20.0

# Irrelevant for one particle, but retained so the call is identical to the
# multi-particle API.
k_contact = 2.0e5
gamma_contact = 20.0

pad = radius + particle_radius + 2.0
box = np.array([-pad, pad, -pad, pad], dtype=np.float32)

## Run the Euclidean solver

In [ ]:
t_e, x_e, v_e = simulate_euclidean_particles(
    box=box,
    x0=x0,
    v0=v0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=T_max,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    radius=radius,
    xA=xA,
    omega=omega,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    save_every=save_every,
)

x_e_1 = x_e[:, 0, :]
v_e_1 = v_e[:, 0, :]

t_e.shape, x_e_1.shape

## Run the polar solver

In [ ]:
q0, p0 = euclidean_to_polar_particles(x0, v0)

t_p, q_p, p_p, x_from_polar, v_from_polar = simulate_polar_particles(
    box=box,
    q0=q0,
    p0=p0,
    m=m,
    rad=rad,
    dt=dt,
    T_max=T_max,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    radius=radius,
    xA=xA,
    omega=omega,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    save_every=save_every,
    r_cap=0.05,
)

q_p_1 = q_p[:, 0, :]
p_p_1 = p_p[:, 0, :]
x_p_1 = x_from_polar[:, 0, :]
v_p_1 = v_from_polar[:, 0, :]

t_p.shape, q_p_1.shape, x_p_1.shape

## Trajectory in the polar coordinate box

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

theta_plot = q_p_1[:, 1].copy()
r_plot = q_p_1[:, 0].copy() / radius

# Break the line when theta wraps, so matplotlib does not draw through the seam.
theta_line = [theta_plot[0]]
r_line = [r_plot[0]]

for i in range(1, len(theta_plot)):
    if abs(theta_plot[i] - theta_plot[i - 1]) > np.pi:
        theta_line.append(np.nan)
        r_line.append(np.nan)

    theta_line.append(theta_plot[i])
    r_line.append(r_plot[i])

ax.plot(theta_line, r_line, lw=1.5, label="polar trajectory")

ax.scatter(
    [q0[0, 1]],
    [q0[0, 0] / radius],
    s=45,
    zorder=4,
    label="initial position",
)

ax.axhline(
    1.0,
    color="black",
    ls="--",
    lw=1.0,
    label="wall position",
)

ax.set_xlim(0.0, 2.0 * np.pi)
ax.set_ylim(0.0, 1.1)
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$r/R$")
ax.set_title("Polar trajectory in coordinate space")
ax.legend(loc="lower right")

#fig.savefig("my_figure.png", dpi=300, bbox_inches="tight")
plt.show()

## Euclidean trajectory vs polar trajectory mapped back

In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 500)
wall_x = radius * np.cos(theta)
wall_y = radius * np.sin(theta)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(wall_x, wall_y, color="black", lw=1.0, label="wall")
ax.plot(x_e_1[:, 0], x_e_1[:, 1], lw=1.8, label="Euclidean integration")
ax.plot(x_p_1[:, 0], x_p_1[:, 1], "--", lw=1.5, label="Polar mapped back")
ax.scatter(x0[:, 0], x0[:, 1], s=35, zorder=4, label="initial position")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-radius - 1.0, radius + 1.0)
ax.set_ylim(-radius - 1.0, radius + 1.0)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Single-particle trajectory: Euclidean vs polar")
ax.legend(loc="upper right")
plt.show()

## Coordinate-system difference

In [ ]:
assert np.allclose(t_e, t_p)

position_error = np.linalg.norm(x_e_1 - x_p_1, axis=1)
velocity_error = np.linalg.norm(v_e_1 - v_p_1, axis=1)

position_error_percent = 100.0 * position_error / radius

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_e, position_error_percent)
ax.set_xlabel("t")
ax.set_ylabel("position error [% of radius]")
ax.set_title("Difference between Euclidean and mapped-back polar trajectories")
plt.show()

print(f"max position error: {position_error.max():.6e}")
print(f"max position error: {position_error_percent.max():.6e}% of radius")
print(f"max velocity error: {velocity_error.max():.6e}")

## Timestep check
 
 This sweep is intentionally short. For this float32 validation solver, the useful check is whether the coordinate-system difference decreases across ordinary DEM timesteps. Very small timesteps over very long integrations can increase error because the number of float32 updates becomes enormous.

In [ ]:
dt_values = [1.0e-3, 7.50e-4, 5.0e-4, 2.50e-4, 1.0e-4]
truth_dt = 1.0e-4
sweep_T_max = 4.0


def run_euclidean(dt_i):
    save_every_i = max(1, int(round(1.0e-2 / dt_i)))
    return simulate_euclidean_particles(
        box=box,
        x0=x0,
        v0=v0,
        m=m,
        rad=rad,
        dt=dt_i,
        T_max=sweep_T_max,
        group_mobile=group_mobile,
        k_contact=k_contact,
        gamma_contact=gamma_contact,
        radius=radius,
        xA=xA,
        omega=omega,
        k_w=k_w,
        gamma_w=gamma_w,
        g=g,
        save_every=save_every_i,
    )


def run_polar(dt_i):
    save_every_i = max(1, int(round(1.0e-2 / dt_i)))
    q0_i, p0_i = euclidean_to_polar_particles(x0, v0)
    return simulate_polar_particles(
        box=box,
        q0=q0_i,
        p0=p0_i,
        m=m,
        rad=rad,
        dt=dt_i,
        T_max=sweep_T_max,
        group_mobile=group_mobile,
        k_contact=k_contact,
        gamma_contact=gamma_contact,
        radius=radius,
        xA=xA,
        omega=omega,
        k_w=k_w,
        gamma_w=gamma_w,
        g=g,
        save_every=save_every_i,
        r_cap=0.05,
    )


def interp_xy(t_ref, x_ref, t_query):
    out = np.empty((len(t_query), 2), dtype=float)
    out[:, 0] = np.interp(t_query, t_ref, x_ref[:, 0])
    out[:, 1] = np.interp(t_query, t_ref, x_ref[:, 1])
    return out


# Warm up compiled kernels so timing does not include Numba compilation.
_ = run_euclidean(1.0e-3)
_ = run_polar(1.0e-3)

sweep = {}
for dt_i in dt_values:
    tic = time.perf_counter()
    t_e_i, x_e_i, v_e_i = run_euclidean(dt_i)
    euclidean_time = time.perf_counter() - tic

    tic = time.perf_counter()
    t_p_i, q_p_i, p_p_i, x_p_i, v_p_i = run_polar(dt_i)
    polar_time = time.perf_counter() - tic

    sweep[dt_i] = {
        "t_e": t_e_i,
        "x_e": x_e_i[:, 0, :].astype(float),
        "t_p": t_p_i,
        "x_p": x_p_i[:, 0, :].astype(float),
        "euclidean_time": euclidean_time,
        "polar_time": polar_time,
    }

truth = sweep[truth_dt]
x_e_truth = truth["x_e"]
t_e_truth = truth["t_e"]
x_p_truth = truth["x_p"]
t_p_truth = truth["t_p"]
time_ref = sweep[1.0e-3]["euclidean_time"]

rows = []
for dt_i in dt_values:
    item = sweep[dt_i]

    x_p_on_e_time = interp_xy(item["t_p"], item["x_p"], item["t_e"])
    coordinate_error = np.linalg.norm(item["x_e"] - x_p_on_e_time, axis=1).max()

    x_e_truth_on_dt = interp_xy(t_e_truth, x_e_truth, item["t_e"])
    euclidean_degradation = np.linalg.norm(item["x_e"] - x_e_truth_on_dt, axis=1).max()

    x_p_truth_on_dt = interp_xy(t_p_truth, x_p_truth, item["t_p"])
    polar_degradation = np.linalg.norm(item["x_p"] - x_p_truth_on_dt, axis=1).max()

    rows.append({
        "dt": dt_i,
        "coordinate_error": coordinate_error,
        "euclidean_degradation": euclidean_degradation,
        "polar_degradation": polar_degradation,
        "euclidean_time": item["euclidean_time"],
        "polar_time": item["polar_time"],
        "euclidean_time_ratio": item["euclidean_time"] / time_ref,
        "polar_time_ratio": item["polar_time"] / time_ref,
    })

print(
    "dt        coord err    E worse vs 1e-4   P worse vs 1e-4   "
    "E time ratio   P time ratio"
)
for row in rows:
    print(
        f"{row['dt']:.2e}  "
        f"{row['coordinate_error']:.6e}  "
        f"{row['euclidean_degradation']:.6e}     "
        f"{row['polar_degradation']:.6e}     "
        f"{row['euclidean_time_ratio']:.3f}          "
        f"{row['polar_time_ratio']:.3f}"
    )

## Sweep plots


In [ ]:
dt_plot = np.array([row["dt"] for row in rows])
coordinate_error = np.array([row["coordinate_error"] for row in rows])
euclidean_time_ratio = np.array([row["euclidean_time_ratio"] for row in rows])
polar_time_ratio = np.array([row["polar_time_ratio"] for row in rows])

reference_dt = 1.0e-4
x_truth = sweep[reference_dt]["x_e"]
t_truth = sweep[reference_dt]["t_e"]

baseline = coordinate_error[dt_plot == reference_dt][0]

euclidean_error_ratio = []
polar_error_ratio = []

for dt_i in dt_plot:
    item = sweep[dt_i]

    x_truth_on_e_time = interp_xy(t_truth, x_truth, item["t_e"])
    e_err = np.linalg.norm(item["x_e"] - x_truth_on_e_time, axis=1).max()

    x_truth_on_p_time = interp_xy(t_truth, x_truth, item["t_p"])
    p_err = np.linalg.norm(item["x_p"] - x_truth_on_p_time, axis=1).max()

    euclidean_error_ratio.append(e_err / baseline)
    polar_error_ratio.append(p_err / baseline)

euclidean_error_ratio = np.array(euclidean_error_ratio)
polar_error_ratio = np.array(polar_error_ratio)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(dt_plot, coordinate_error, marker="o")
ax.invert_xaxis()
ax.set_xlabel("dt")
ax.set_ylabel("max position difference")
ax.set_title("Euclidean vs polar at the same timestep")
plt.show()

dt_plot = np.array([row["dt"] for row in rows])

reference_dt = 1.0e-4
t_ref = sweep[reference_dt]["t_e"]
x_ref = sweep[reference_dt]["x_e"]

euclidean_relative_error = []
polar_relative_error = []

for dt_i in dt_plot:
    item = sweep[dt_i]

    # Euclidean solver at dt_i compared with Euclidean dt=1e-4 reference.
    x_ref_on_e_time = interp_xy(t_ref, x_ref, item["t_e"])
    e_num = np.linalg.norm(item["x_e"] - x_ref_on_e_time, axis=1)

    # Polar solver at dt_i compared with Euclidean dt=1e-4 reference.
    x_ref_on_p_time = interp_xy(t_ref, x_ref, item["t_p"])
    p_num = np.linalg.norm(item["x_p"] - x_ref_on_p_time, axis=1)

    # Normalize by the magnitude of the reference trajectory.
    # Add a tiny epsilon to avoid division problems if the particle passes close to the origin.
    e_den = np.linalg.norm(x_ref_on_e_time, axis=1)
    p_den = np.linalg.norm(x_ref_on_p_time, axis=1)

    eps = 1.0e-12
    euclidean_relative_error.append(np.max(e_num / (e_den + eps)))
    polar_relative_error.append(np.max(p_num / (p_den + eps)))

euclidean_relative_error = np.array(euclidean_relative_error)
polar_relative_error = np.array(polar_relative_error)

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(
    dt_plot,
    euclidean_relative_error,
    marker="o",
    label="Euclidean vs Euclidean 1e-4",
)
ax.semilogx(
    dt_plot,
    polar_relative_error,
    marker="s",
    label="Polar vs Euclidean 1e-4",
)
ax.invert_xaxis()
ax.set_xlabel("dt")
ax.set_ylabel(r"relative error $\|x - x_{\rm ref}\| / \|x_{\rm ref}\|$")
ax.set_title("Relative error using Euclidean dt=1e-4 as reference")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(dt_plot, euclidean_time_ratio, marker="o", label="Euclidean")
ax.semilogx(dt_plot, polar_time_ratio, marker="s", label="Polar")
ax.axhline(1.0, color="black", ls="--", lw=1.0, label="Euclidean dt=1e-3 reference")
ax.invert_xaxis()
ax.set_xlabel("dt")
ax.set_ylabel("runtime ratio")
ax.set_title("Computational time relative to Euclidean dt=1e-3")
ax.legend()
plt.show()

## Segment-boundary resolution study
 
 This test keeps the same single-particle physical problem and compares the analytic circular wall in `euclidean_solver_v1.py` with the explicit segment wall in `euclidean_solver_v2.py`. The circle is approximated by increasingly many connected boundary segments. The red marker indicates the first resolution where the explicit segment-wall runtime becomes larger than the polar solver runtime at the same conditions.

In [ ]:
from euclidean_solver_v2 import (
    simulate_euclidean_particles as simulate_segment_boundary_particles,
    make_circle_boundary_nodes,
)

segment_counts = np.array(
    [8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024, 1536, 2048],
    dtype=int,
)
segment_dt = 1.0e-4
segment_T_max = T_max
segment_save_every = save_every

# Use the analytic Euclidean circle solver as the reference trajectory.
t_ref = t_e
x_ref = x_e_1.astype(float)

# Warm up both compiled paths. These short runs are only to trigger Numba JIT;
# they are deliberately excluded from the measured timings below.
q0_warm, p0_warm = euclidean_to_polar_particles(x0, v0)
_ = simulate_polar_particles(
    box=box,
    q0=q0_warm,
    p0=p0_warm,
    m=m,
    rad=rad,
    dt=segment_dt,
    T_max=segment_dt,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    radius=radius,
    xA=xA,
    omega=omega,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    save_every=1,
    r_cap=0.05,
)
_ = simulate_segment_boundary_particles(
    box=box,
    x0=x0,
    v0=v0,
    m=m,
    rad=rad,
    dt=segment_dt,
    T_max=segment_dt,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    boundary_nodes=make_circle_boundary_nodes(radius, 8, closed=True),
    xA=xA,
    omega=omega,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    save_every=1,
)

# Time the polar solver at the same conditions. This is the runtime threshold
# used to identify when the explicit segment wall becomes more expensive.
q0_ref, p0_ref = euclidean_to_polar_particles(x0, v0)
tic = time.perf_counter()
t_p_cost, q_p_cost, p_p_cost, x_p_cost, v_p_cost = simulate_polar_particles(
    box=box,
    q0=q0_ref,
    p0=p0_ref,
    m=m,
    rad=rad,
    dt=segment_dt,
    T_max=segment_T_max,
    group_mobile=group_mobile,
    k_contact=k_contact,
    gamma_contact=gamma_contact,
    radius=radius,
    xA=xA,
    omega=omega,
    k_w=k_w,
    gamma_w=gamma_w,
    g=g,
    save_every=segment_save_every,
    r_cap=0.05,
)
polar_same_condition_time = time.perf_counter() - tic

segment_rows = []
for n_segments in segment_counts:
    boundary_nodes = make_circle_boundary_nodes(radius, n_segments, closed=True)

    tic = time.perf_counter()
    t_seg, x_seg, v_seg = simulate_segment_boundary_particles(
        box=box,
        x0=x0,
        v0=v0,
        m=m,
        rad=rad,
        dt=segment_dt,
        T_max=segment_T_max,
        group_mobile=group_mobile,
        k_contact=k_contact,
        gamma_contact=gamma_contact,
        boundary_nodes=boundary_nodes,
        xA=xA,
        omega=omega,
        k_w=k_w,
        gamma_w=gamma_w,
        g=g,
        save_every=segment_save_every,
    )
    elapsed = time.perf_counter() - tic

    x_seg_1 = x_seg[:, 0, :].astype(float)
    x_ref_on_seg_time = interp_xy(t_ref, x_ref, t_seg)
    position_error = np.linalg.norm(x_seg_1 - x_ref_on_seg_time, axis=1)

    segment_rows.append({
        "segments": int(n_segments),
        "max_error": float(position_error.max()),
        "max_error_percent": float(100.0 * position_error.max() / radius),
        "rms_error_percent": float(100.0 * np.sqrt(np.mean(position_error**2)) / radius),
        "time": float(elapsed),
        "time_ratio_to_polar": float(elapsed / polar_same_condition_time),
    })

print(f"polar solver time at same conditions, excluding JIT: {polar_same_condition_time:.6f} s")
print("segments   max error [%R]   rms error [%R]   segment time [s]   time / polar")
for row in segment_rows:
    print(
        f"{row['segments']:8d}   "
        f"{row['max_error_percent']:14.6e}   "
        f"{row['rms_error_percent']:14.6e}   "
        f"{row['time']:16.6f}   "
        f"{row['time_ratio_to_polar']:12.3f}"
    )

In [ ]:
segments = np.array([row["segments"] for row in segment_rows])
max_error_percent = np.array([row["max_error_percent"] for row in segment_rows])
time_ratio_to_polar = np.array([row["time_ratio_to_polar"] for row in segment_rows])

# Stop the plotted comparison at 1024 polygon segments.
keep = segments <= 1024

segments_plot = segments[keep]
max_error_percent_plot = max_error_percent[keep]
time_ratio_to_polar_plot = time_ratio_to_polar[keep]

slower = np.nonzero(time_ratio_to_polar_plot >= 1.0)[0]
cross_idx = int(slower[0]) if slower.size else None

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(
    segments_plot,
    max_error_percent_plot,
    marker="o",
    label="max trajectory error",
)

if cross_idx is not None:
    ax.scatter(
        [segments_plot[cross_idx]],
        [max_error_percent_plot[cross_idx]],
        s=90,
        color="red",
        zorder=5,
        label="segment wall slower than polar",
    )

ax.set_xlabel("number of boundary segments")
ax.set_ylabel("max error relative to analytic circle [% of radius]")
ax.set_title("Boundary resolution vs trajectory error")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(
    segments_plot,
    time_ratio_to_polar_plot,
    marker="o",
)

ax.axhline(
    1.0,
    color="red",
    ls="--",
    lw=1.0,
    label="polar solver runtime",
)

ax.set_xlabel("number of boundary segments")
ax.set_ylabel("runtime / polar runtime")
ax.set_title("Explicit segment-wall cost relative to polar solver")
ax.legend()
plt.show()